In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

print(v1.dot(dv))

q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

print(v2.dot(dv))

0.323324
0.019730432


In [5]:
from ingest import load_faq_data

documents = load_faq_data()

In [6]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)



# we bacth the vectorisation

from tqdm.auto import tqdm
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1400

In [7]:
import numpy as np
X = np.array(vectors)

print(f"number of documents: {X.shape[0]}")
print(f"number of dimentions: {X.shape[1]}")

number of documents: 1400
number of dimentions: 384


In [12]:
# scoring documents

query = "Can I still join the course after the start date?"
v_query = model.encode(query)


# cosine similarity between document i and v_query
scores = X.dot(v_query)

# identify the best match


idx = np.argmax(scores)
print(f"document {idx} with a score: {scores[idx]}")

documents[510]

document 510 with a score: 0.7629411816596985


{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [13]:
# important to negate the scores, because argsort sorts ASC
top5 = np.argsort(-scores)[:5]

for idx in top5:
    print(scores[idx])
    print(documents[idx])

0.7629412
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}
0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}
0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Q

In [ ]:
# same but now using min search and filter feature
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

query = "I just discovered the course. Can I still join it?"
v_query = model.encode(query)

results = vindex.search(
    v_query,
    filter_dict = {"course": "llm-zoomcamp"},
    num_results=5
)

print(results[0])

{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}


In [17]:
# Now let's implement a RAG with Vector Search

from rag_helper import RAGBase


class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder


    def search(self, query, num_results=5):
        v_query = self.embedder.encode(query)
        return self.index.search(
            v_query,
            filter_dict = {"course": self.course},
            num_results = num_results
        )


import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

vector_assistant = RAGVector(
    embedder=model,
    index = vindex,
    llm_client=openai_client,
    model=os.getenv("OPENAI_MODEL")
)

vector_assistant.rag("the program has already begun, can I still sign up?")

'\n\nYes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still open. Additionally, you can start learning and submitting homework without registering, as registration is primarily to gauge interest before the start date.'

In [ ]:
# Using persistant index and ANN (approximate search by region, faster)

from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf", # up to 100K documents, fine for our small dataset
    db_path="faq_vectors.db"
)

vs_index.fit(vectors, documents)


ValueError: Index already contains documents. Use clear() to reset the index or add() to append documents.

In [25]:
results = vs_index.search(
    v_query,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

print(results[0])

v_query = model.encode("How do I run Kafka?")
results = vs_index.search(v_query, num_results=5)

print(results)

vs_index.close()


{'id': '04440cab11', 'course': 'llm-zoomcamp', 'section': 'Module 3: Orchestration', 'question': 'Kestra AI Copilot replies "I can only assist with creating Kestra flows" — how do I fix it?', 'answer': 'This message means the AI Copilot didn\'t get a valid Gemini API key, so it falls back to a canned refusal. In Kestra\'s Open Source edition the Copilot only supports Gemini, and it reads the **plain** `GEMINI_API_KEY` variable (not the base64-encoded `SECRET_GEMINI_API_KEY` that the flows use).\n\nMake sure you exported the plain key before starting Kestra, then restart it:\n\n```bash\nexport GEMINI_API_KEY="your-gemini-api-key-here"\ndocker compose up -d\n```\n\nIf it still fails, the key is usually missing, mistyped, or rate-limited:\n\n- Confirm the variable is actually set in the shell you ran `docker compose up` from (`echo $GEMINI_API_KEY`).\n- Generate a fresh key in [Google AI Studio](https://aistudio.google.com/app/apikey).\n- If you\'ve been running the agent/multi-agent flow